# Scratchpad: `GlmMoeDsaNaiveMoe` I/O

Layer the dependencies so each cell has one job:

1. **Global deps** — packages everything needs (`torch`, `nn`, `F`, `ACT2FN`)
2. **Local deps** — toy config / sizes this class alone reads
3. **The class** — definition only (no call)
4. **Call** — build inputs that match `forward`, run, print shapes

Run top → bottom.

## 1. Global dependencies

Anything that is not about *this* class — shared infrastructure.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.activations import ACT2FN

torch.manual_seed(0)
print("global deps ready:", torch.__version__)

global deps ready: 2.12.0


## 2. Local dependencies (what *this* class reads)

`GlmMoeDsaNaiveMoe` only needs a few config fields — not the full model:

| field | used for |
|---|---|
| `num_local_experts` | how many expert weight slices |
| `hidden_size` | token feature dim |
| `moe_intermediate_size` | expert MLP width |
| `hidden_act` | activation name → `ACT2FN[...]` |

In the real config, `num_local_experts` aliases `n_routed_experts`. Here we fake a tiny config object so we don't pull in the whole `GlmMoeDsaConfig`.

In [2]:
from types import SimpleNamespace

# toy sizes — small enough to print by hand
NUM_EXPERTS = 4
HIDDEN = 8
INTERMEDIATE = 16
TOP_K = 2          # how many experts each token hits (router would choose this)
NUM_TOKENS = 6     # flattened B*S tokens

config = SimpleNamespace(
    num_local_experts=NUM_EXPERTS,
    hidden_size=HIDDEN,
    moe_intermediate_size=INTERMEDIATE,
    hidden_act="silu",
)

print("local config:", config.__dict__)

local config: {'num_local_experts': 4, 'hidden_size': 8, 'moe_intermediate_size': 16, 'hidden_act': 'silu'}


## 3. The class

Same code as `david/architecture/modeling_glm_moe_dsa.py` — pasted here so you can see every name it closes over (`nn`, `F`, `ACT2FN`, `config` fields).

In [3]:
class GlmMoeDsaNaiveMoe(nn.Module):
    """Collection of expert weights stored as 3D tensors — same math as GlmMoeDsaMLP per expert."""

    def __init__(self, config):
        super().__init__()
        self.num_experts = config.num_local_experts
        self.hidden_dim = config.hidden_size
        self.intermediate_dim = config.moe_intermediate_size
        self.up_proj = nn.Parameter(torch.empty(self.num_experts, self.intermediate_dim, self.hidden_dim))
        self.down_proj = nn.Parameter(torch.empty(self.num_experts, self.hidden_dim, self.intermediate_dim))
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = F.one_hot(top_k_index, num_classes=self.num_experts)
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            current_hidden_states = self.act_fn(F.linear(current_state, self.up_proj[expert_idx]))
            current_hidden_states = F.linear(current_hidden_states, self.down_proj[expert_idx])
            current_hidden_states = current_hidden_states * top_k_weights[token_idx, top_k_pos, None]
            final_hidden_states.index_add_(0, token_idx, current_hidden_states.to(final_hidden_states.dtype))

        return final_hidden_states

print("class defined:", GlmMoeDsaNaiveMoe)

class defined: <class '__main__.GlmMoeDsaNaiveMoe'>


## 4. Call the class (I/O contract)

`forward` expects three tensors (already flattened to tokens — MoE parent does the reshape):

| arg | shape | meaning |
|---|---|---|
| `hidden_states` | `(T, H)` | T tokens, each hidden_size |
| `top_k_index` | `(T, K)` | which expert ids each token uses |
| `top_k_weights` | `(T, K)` | mix weights for those experts |

Returns `(T, H)` — same shape as input, sparse expert outputs summed in.

In [4]:
experts = GlmMoeDsaNaiveMoe(config)

# real models init after empty(); do it here so we don't feed garbage weights
nn.init.normal_(experts.up_proj, mean=0.0, std=0.02)
nn.init.normal_(experts.down_proj, mean=0.0, std=0.02)

print("up_proj  ", tuple(experts.up_proj.shape),   "← (E, I, H)")
print("down_proj", tuple(experts.down_proj.shape), "← (E, H, I)")

# --- inputs matching forward's contract ---
hidden_states = torch.randn(NUM_TOKENS, HIDDEN)                      # (T, H)

# fake router choices: each token picks TOP_K distinct experts
top_k_index = torch.stack([
    torch.randperm(NUM_EXPERTS)[:TOP_K] for _ in range(NUM_TOKENS)
])                                                                   # (T, K)

top_k_weights = torch.softmax(torch.randn(NUM_TOKENS, TOP_K), dim=-1)  # (T, K), sum→1

print("\nhidden_states", tuple(hidden_states.shape))
print("top_k_index  ", tuple(top_k_index.shape), "\n", top_k_index)
print("top_k_weights", tuple(top_k_weights.shape))

out = experts(hidden_states, top_k_index, top_k_weights)

print("\nout         ", tuple(out.shape), "← same as hidden_states")
print("out[0] sample", out[0].detach())
assert out.shape == hidden_states.shape
print("PASS: output shape matches input")

up_proj   (4, 16, 8) ← (E, I, H)
down_proj (4, 8, 16) ← (E, H, I)

hidden_states (6, 8)
top_k_index   (6, 2) 
 tensor([[1, 0],
        [1, 0],
        [0, 3],
        [1, 0],
        [0, 3],
        [2, 1]])
top_k_weights (6, 2)

out          (6, 8) ← same as hidden_states
out[0] sample tensor([ 0.0019,  0.0014,  0.0023,  0.0003, -0.0025,  0.0039,  0.0007, -0.0024])
PASS: output shape matches input


## 5. (optional) Trace one expert

Which tokens hit expert 0? Confirms the dispatch loop is doing something sparse.

In [5]:
expert_id = 0
hits = (top_k_index == expert_id).nonzero(as_tuple=False)
print(f"expert {expert_id} hit by {hits.shape[0]} (token, k_slot) pairs:")
print(hits)

# tokens that never chose expert 0 still get a contribution from *other* experts;
# rows that only used experts that were never hit would stay zero — with random
# routing + TOP_K>=1 every token should be nonzero almost always.
print("\n||out|| per token:", out.norm(dim=-1).detach())

expert 0 hit by 5 (token, k_slot) pairs:
tensor([[0, 1],
        [1, 1],
        [2, 0],
        [3, 1],
        [4, 0]])

||out|| per token: tensor([0.0062, 0.0034, 0.0035, 0.0050, 0.0054, 0.0069])


## 6. Where `router_logits` comes from

`NaiveMoe` never sees router logits — the **gate** produces them, then `route_tokens_to_experts` turns them into `top_k_index` / `top_k_weights`.

This notebook skipped that and faked those two tensors. To play with the first line of routing (`sigmoid`), build logits yourself:

In [8]:
# gate output shape: (T, E) — one score per token per expert
router_logits = torch.randn(NUM_TOKENS, NUM_EXPERTS)
print("router_logits", tuple(router_logits.shape))
print(router_logits)

# first step of route_tokens_to_experts — note: .sigmoid() not .sigmoind()
print("\nafter sigmoid:")
print(router_logits.sigmoid())

router_logits (6, 4)
tensor([[ 0.1385, -0.1659,  1.2669, -0.4962],
        [ 0.1581,  0.4889, -0.3439,  0.2291],
        [-1.5185,  0.0475, -0.0103,  1.4353],
        [ 0.2518,  0.7408,  1.4431, -0.2053],
        [-1.3989,  0.0408, -1.1174,  0.7029],
        [-0.6675,  0.1254,  0.7900, -0.0095]])

after sigmoid:
tensor([[0.5346, 0.4586, 0.7802, 0.3784],
        [0.5394, 0.6198, 0.4149, 0.5570],
        [0.1797, 0.5119, 0.4974, 0.8077],
        [0.5626, 0.6772, 0.8089, 0.4488],
        [0.1980, 0.5102, 0.2465, 0.6688],
        [0.3391, 0.5313, 0.6878, 0.4976]])
